# 0. Setup 
--- 

## Imports

In [1]:
# Standard library
import argparse
import json
import os
import random
import sys
from pathlib import Path
from time import time
from typing import Any, Dict, List, Optional, Tuple

# Third-party libraries
import numpy as np
import pandas as pd
import torch
from torch.nn.functional import cosine_similarity
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

import nnsight
from nnsight import LanguageModel

# Local/project imports
from modsteer import PROJECT_ROOT
from modsteer.dataset.load_dataset import load_dataset_split
from modsteer.steering.utils import (
    to_chat as _to_chat,
    compute_mean_activations as _mean_acts_for_batched,
    diffmean as _diffmean_from_means,
)
from modsteer.steering.utils import diffmean
from modsteer.dataset.load_dataset import load_ifeval_pairs
from modsteer.utils import set_seed

import hydra
from hydra import initialize, compose
from typing import Dict, List


## Boilerplate

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
set_seed(42)

## Configuration

In [4]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path=str("../config/"), job_name="innocuous_steering")
cfg = compose(config_name="steering", overrides=[])

## Load model and tokenizer

In [5]:
model = LanguageModel(
    cfg.model,
    device_map=cfg.device,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(cfg.model, trust_remote_code=True)
tokenizer.padding_side = 'left'

# Manually attach the config from the Hugging Face model to the nnsight model
# wrapper, as this is not done automatically when initializing from an object.
# model.config = hf_model.config
# Reduce memory by disabling KV cache during traces
model.model.config.use_cache = False

In [6]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model)
tokenizer.padding_side = 'left'

if tokenizer.pad_token is None:
    print('Setting pad_token to eos_token')
    tokenizer.pad_token = tokenizer.eos_token

In [7]:
# Freeze model params to avoid accumulating grads during inputs_embeds optimization
# model.model for gemma-2-2b-it
for p in model.parameters():
    p.requires_grad_(False)

# 1. Compute refusal steering vectors 
---

## Data loading

In [8]:
harmful_train = random.sample(load_dataset_split(harmtype='harmful', split='train', instructions_only=True), cfg.num_train_samples)
harmless_train = random.sample(load_dataset_split(harmtype='harmless', split='train', instructions_only=True), cfg.num_train_samples)


## Compute steering vectors

In [9]:
print(f"Will extract activations from the last {len(cfg.token_positions)} token positions: {cfg.token_positions}")

# (num_layers, num_token_positions, embedding_dim)
harmless_to_harmful_directions: torch.Tensor = diffmean(
    model=model,
    tokenizer=tokenizer,
    positive_prompts=harmful_train,
    negative_prompts=harmless_train,
    device=cfg.device,
    batch_size=16,
) 

import os

# Ensure the output directory exists
os.makedirs(os.path.dirname(cfg.refusal_directions_path), exist_ok=True)

torch.save(harmless_to_harmful_directions, cfg.refusal_directions_path)
print(f"POST directions computed and saved to {cfg.refusal_directions_path}")

You have set `use_cache` to `False`, but cache_implementation is set to hybrid. cache_implementation will have no effect.


Will extract activations from the last 1 token positions: [-1]
End-of-Instruction tokens decoded: <end_of_turn>
<start_of_turn>model

Using token positions [-6, -5, -4, -3, -2, -1] for mean activation computation


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

End-of-Instruction tokens decoded: <end_of_turn>
<start_of_turn>model

Using token positions [-6, -5, -4, -3, -2, -1] for mean activation computation
POST directions computed and saved to /workspace/adversarial_attack/stored_vectors/refusal_directions_gemma-2-2b-it.pt


In [10]:
refusal_directions = torch.load(cfg.refusal_directions_path)

In [11]:
tok_poses = [-1]
for tok_pos in tok_poses:
    torch.save(refusal_directions[:, tok_pos, :], f"/workspace/adversarial_attack/stored_vectors/refusal_directions_{cfg.model.split('/')[-1]}_tokpos_{tok_pos}.pt")


# 2. Compute Innocuous Steering Vectors
---

### Load instruction following dataset

In [13]:
pairs_by_attribute = {}

attribute_to_id = {
    # "uppercase": "change_case:english_capital",
    "lowercase": "change_case:english_lowercase",
    "no_comma": "punctuation:no_comma",
    # "json": "format:json",
    # "title_fmt": "format:title",
    "emoji": "format:emoji"
}

In [14]:
for attribute in cfg.attributes:
    if attribute == "emoji":
        # Load custom emoji dataset
        positive, negative = load_ifeval_pairs(
            data_path="/workspace/adversarial_attack/data/gpt_generations/emoji_pairs.jsonl",
            num_pairs=20
        )
    else:
        # Load from IFEval dataset
        positive, negative = load_ifeval_pairs(
            data_path=cfg.instruct_following_data_path,
            num_pairs=50,
            instruction_id=attribute_to_id[attribute]
        )
    
    pairs_by_attribute[attribute] = (positive, negative)


## Compute steering vectors

In [15]:
# negative -> positive is the direction from format-free prompts to instruction-following prompts, e.g. standard prompt -> UPPERCASE PROMPT

negative_to_positive_directions = {}

for attribute in cfg.attributes:

    positive, negative = pairs_by_attribute[attribute]

    # (num_layers, num_token_positions, embedding_dim)
    negative_to_positive_directions[attribute] = diffmean(
        model=model,
        tokenizer=tokenizer,
        positive_prompts=positive,
        negative_prompts=negative,
        device=cfg.device,
        batch_size=16,
    )

    path_name = lambda attr: cfg.instruct_directions_path.replace(".pt", f"_{attr}.pt")
    torch.save(negative_to_positive_directions[attribute], path_name(attribute))
    print(f"instruct directions computed and saved to {path_name(attribute)}")

End-of-Instruction tokens decoded: <end_of_turn>
<start_of_turn>model

Using token positions [-6, -5, -4, -3, -2, -1] for mean activation computation
End-of-Instruction tokens decoded: <end_of_turn>
<start_of_turn>model

Using token positions [-6, -5, -4, -3, -2, -1] for mean activation computation
instruct directions computed and saved to /workspace/adversarial_attack/stored_vectors/innocuous_directions_gemma-2-2b-it_emoji.pt


In [16]:
tok_poses = [-1]
for attribute in cfg.attributes:
    for tok_pos in tok_poses:
        torch.save(negative_to_positive_directions[attribute][:, tok_pos, :], f"/workspace/adversarial_attack/stored_vectors/innocuous_directions_{cfg.model.split('/')[-1]}_{attribute}_{tok_pos}.pt")

# 3. Compare steering vectors
---

In [ ]:
chosen_harmless_to_harmful_direction = harmless_to_harmful_directions[cfg.refusal_direction_layer][cfg.refusal_direction_token_position]
cosine_similarities = []

for attribute in cfg.attributes:

    chosen_free_to_instruct_direction = negative_to_positive_directions[attribute][cfg.instruct_direction_layer][cfg.instruct_direction_token_position]

    cos_sim = cosine_similarity(
        chosen_harmless_to_harmful_direction.unsqueeze(0),
        chosen_free_to_instruct_direction.unsqueeze(0)
    ).item()
    cosine_similarities.append((attribute, cos_sim))


In [ ]:
cosine_similarities

In [ ]:
# plot cosine similarities
import matplotlib.pyplot as plt

attributes, similarities = zip(*cosine_similarities)
plt.bar(attributes, similarities)
plt.ylabel('Cosine Similarity')
plt.xlabel('Attribute')
plt.title('Cosine Similarity between Harmless->Harmful and Free->Instruct Directions')

plt.show()
